# Fase 3. Preparación de los datos

La preparación de los datos adapta la información para que pueda utilizarse en las técnicas de
minería de datos. Esta fase se desarrolla en dos partes.

En la primera parte, secciones 3.1 a 3.8, se realizan la selección de los datos (qué columnas y
registros se utilizan), el formateo (ajuste de nombres y tipos sin cambiar el significado de los
datos) y la limpieza (corrección de los problemas de calidad encontrados en la sección 2).

En la segunda parte, a partir de la sección 3.9, se construyen las variables por unidad y se
transforman y estandarizan para el modelado.

Cada corrección se presenta en tres pasos: cómo estaban los datos antes, qué cambio se aplicó y cómo
quedaron después. El archivo original en data/raw/ no se modifica; el resultado se guarda como un
archivo nuevo en data/processed/.

Las correcciones se basan en los hallazgos de la sección 2.5:

| Hallazgo de la exploración | Acción |
|---|---|
| Fecha de actualización tiene un único valor | Se excluye |
| Nombre es un identificador personal que no aporta al análisis (y tiene 29 registros con espacios sobrantes) | Se excluye |
| No hay valores nulos | Se confirma; no se requiere imputación |
| Nombres de columnas con espacios, tildes y paréntesis | Se renombran |
| Cedula y el código de unidad se leen como números | Se convierten a texto |
| 30 puestos con espacios al final y 243 con espacios dobles | Se eliminan los espacios sobrantes |
| 1 registro con Puesto = #N/D | Se elimina |
| Nombre repetido, cédulas de 11 dígitos y puestos sin término de jefatura | Se conservan y se documentan |

## 3.1 Carga de los datos

La preparación parte del archivo original, que se vuelve a cargar para trabajar sobre los datos sin ninguna modificación previa.

In [21]:
# Librerías utilizadas en esta fase
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

El archivo se carga con encoding="latin-1" porque, con la codificación por defecto de pandas
(UTF-8), las tildes y la ñ no se leen correctamente.

In [22]:
ruta_archivo = "../data/raw/MD1_CL_jefaturas_export_19112026.csv"

df = pd.read_csv(
    ruta_archivo,
    sep=";",
    encoding="latin-1"
)

print("Registros:", df.shape[0])
print("Columnas:", df.shape[1])
df.head()

Registros: 1949
Columnas: 6


,Cedula,Nombre,Unidad de procedencia (UP),Detalle de UP,Puesto,Fecha de actualización
0,107790683,KARLA SOLANO DURAN,1100,Gerencia General,Jefe Despacho Gerencia General,2026-10-12 00:00:00
1,113570358,JOSE LUIS ROJAS HENRIQUEZ,1101,Junta Directiva,Jefe Subárea de Sede,2026-10-12 00:00:00
2,113750446,MARIA BEATRIZ GUZMAN MEZA,1101,Junta Directiva,Jefe Área de Sede,2026-10-12 00:00:00
3,107960815,LAURA MARIA AVILA BOLAÑOS,1102,Presidencia Ejecutiva,Asesor Coordinar Presidencia Ejecutiva,2026-10-12 00:00:00
4,108340101,ALEXANDER SOLIS ABARCA,1103,Gerencia Financiera,Jefe Área de Sede,2026-10-12 00:00:00


Se trabaja sobre una copia llamada df_limpio, de modo que df conserva los datos originales y se
puede comparar el antes y el después de cada cambio.

In [23]:
df_limpio = df.copy()

registros_iniciales = len(df_limpio)
print("Registros al iniciar la limpieza:", registros_iniciales)

Registros al iniciar la limpieza: 1949


## 3.2 Selección de los datos

En esta tarea se decide qué columnas se conservan para el análisis.

Fecha de actualización se excluye porque todos los registros tienen el mismo valor (12/10/2026). Una columna constante no permite diferenciar unidades.

Nombre se excluye porque es un dato personal que no se utiliza en el análisis. Cada persona ya queda identificada por su cédula, que es única en todos los registros. Al excluir esta columna también queda resuelto el problema de los 29 nombres con espacios sobrantes detectado en la exploración, por lo que no es necesario corregirlos.

Se conservan Cedula, Unidad de procedencia (UP), Detalle de UP y Puesto.

In [24]:
print("Valores distintos en Fecha de actualización:", df_limpio["Fecha de actualización"].nunique())
print("Cédulas repetidas:", df_limpio["Cedula"].duplicated().sum())

Valores distintos en Fecha de actualización: 1
Cédulas repetidas: 0


In [25]:
print("Columnas antes:", df_limpio.columns.tolist())

df_limpio = df_limpio.drop(columns=["Nombre", "Fecha de actualización"])

print("Columnas después:", df_limpio.columns.tolist())

Columnas antes: ['Cedula', 'Nombre', 'Unidad de procedencia (UP)', 'Detalle de UP', 'Puesto', 'Fecha de actualización']
Columnas después: ['Cedula', 'Unidad de procedencia (UP)', 'Detalle de UP', 'Puesto']


## 3.3 Formateo de los datos

### Nombres de las columnas

Los nombres originales tienen espacios, tildes y paréntesis, lo que hace el código más largo y
propenso a errores. Se cambian por nombres cortos, en minúscula y sin espacios. El contenido de las
columnas no cambia.

In [26]:
nuevos_nombres = {
    "Cedula": "cedula",
    "Unidad de procedencia (UP)": "codigo_unidad",
    "Detalle de UP": "unidad",
    "Puesto": "puesto"
}

df_limpio = df_limpio.rename(columns=nuevos_nombres)

df_limpio.columns.tolist()

['cedula', 'codigo_unidad', 'unidad', 'puesto']

### Tipos de datos

pandas leyó cedula y codigo_unidad como números enteros. Sin embargo, ambas son códigos: no tiene
sentido sumarlas ni calcular su promedio. Se convierten a texto para que no se traten como cantidades
por error.

In [27]:
print("Tipos antes:")
print(df_limpio.dtypes)

df_limpio["cedula"] = df_limpio["cedula"].astype(str)
df_limpio["codigo_unidad"] = df_limpio["codigo_unidad"].astype(str)

print()
print("Tipos después:")
print(df_limpio.dtypes)

Tipos antes:
cedula           int64
codigo_unidad    int64
unidad             str
puesto             str
dtype: object

Tipos después:
cedula           str
codigo_unidad    str
unidad           str
puesto           str
dtype: object


## 3.4 Limpieza de los datos

### Valores ausentes

En la exploración no se encontraron valores nulos en ninguna columna, por lo que no es necesario
eliminar registros incompletos ni aplicar técnicas de imputación. Se comprueba de nuevo sobre las
columnas seleccionadas:

In [28]:
df_limpio.isna().sum()

cedula           0
codigo_unidad    0
unidad           0
puesto           0
dtype: int64

### Espacios sobrantes en los textos

En la exploración se encontró que puesto tiene espacios al final y espacios dobles dentro del texto,
por ejemplo Enfermera  4   Licenciada. Aunque en este caso no generan categorías repetidas, se
corrigen para que las tablas y los gráficos se vean limpios y para evitar errores al comparar textos.

La corrección se aplica a puesto y a unidad. En unidad no se encontraron problemas, pero se
incluye para dejar ambas columnas con el mismo tratamiento.

In [29]:
print("Antes de la corrección:")
for columna in ["unidad", "puesto"]:
    texto = df_limpio[columna]
    print(columna)
    print("  Con espacios al inicio o al final:", (texto != texto.str.strip()).sum())
    print("  Con espacios dobles:", texto.str.contains("  ").sum())
    print("  Valores únicos:", texto.nunique())

Antes de la corrección:
unidad
  Con espacios al inicio o al final: 0
  Con espacios dobles: 0
  Valores únicos: 284
puesto
  Con espacios al inicio o al final: 30
  Con espacios dobles: 243
  Valores únicos: 159


In [30]:
puestos_antes = df_limpio["puesto"].copy()

for columna in ["unidad", "puesto"]:
    df_limpio[columna] = (
        df_limpio[columna]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

In [31]:
print("Después de la corrección:")
for columna in ["unidad", "puesto"]:
    texto = df_limpio[columna]
    print(columna)
    print("  Con espacios al inicio o al final:", (texto != texto.str.strip()).sum())
    print("  Con espacios dobles:", texto.str.contains("  ").sum())
    print("  Valores únicos:", texto.nunique())

Después de la corrección:
unidad
  Con espacios al inicio o al final: 0
  Con espacios dobles: 0
  Valores únicos: 284
puesto
  Con espacios al inicio o al final: 0
  Con espacios dobles: 0
  Valores únicos: 159


In [32]:
cambios = pd.DataFrame({
    "Puesto antes": puestos_antes,
    "Puesto después": df_limpio["puesto"]
})

cambios = cambios[cambios["Puesto antes"] != cambios["Puesto después"]]

print("Registros corregidos:", len(cambios))

ejemplos = cambios.drop_duplicates().head(8).copy()
ejemplos["Puesto antes"] = "[" + ejemplos["Puesto antes"] + "]"
ejemplos["Puesto después"] = "[" + ejemplos["Puesto después"] + "]"
ejemplos

Registros corregidos: 273


,Puesto antes,Puesto después
0,[Jefe Despacho Gerencia General ],[Jefe Despacho Gerencia General]
163,[Director Servicios Informáticos ],[Director Servicios Informáticos]
328,[Director Administrativo Financiero Hosp. Nacional ],[Director Administrativo Financiero Hosp. Nacional]
332,[Enfermera 7 Licenciada],[Enfermera 7 Licenciada]
339,[Microbiólogo Clínico 5],[Microbiólogo Clínico 5]
400,[Microbiólogo Clínico 4],[Microbiólogo Clínico 4]
414,[Odontólogo Jefe Servicio ],[Odontólogo Jefe Servicio]
422,[Profesional 1 en Contabilidad y Finanzas ],[Profesional 1 en Contabilidad y Finanzas]


#### Interpretación

Se corrigieron los espacios de 273 registros de puesto. Los corchetes permiten ver que los espacios
sobrantes desaparecieron. La cantidad de puestos distintos se mantiene en 159, lo que confirma que la
corrección solo cambió el formato y no unió ni separó categorías.

### Valor inválido en puesto

Un registro del Hospital de Osa Tomás Casas Casajús tiene el puesto #N/D. Este valor aparece
normalmente cuando una búsqueda en Excel no encuentra el dato, por lo que no corresponde a un puesto
real.

Se decide eliminar el registro porque:

No es posible saber cuál era el puesto real.

Es un solo registro de 1,949 (0.05 %), por lo que su eliminación no afecta el análisis.

Si se conservara, #N/D se contaría como un puesto más del hospital y alteraría sus indicadores de diversidad.

In [33]:
registro_invalido = df_limpio[df_limpio["puesto"] == "#N/D"]

print("Registros con #N/D antes:", len(registro_invalido))
registro_invalido

Registros con #N/D antes: 1


,cedula,codigo_unidad,unidad,puesto
1739,603480987,2703,Hospital de Osa Tomás Casas Casajús,#N/D


In [34]:
unidad_afectada = registro_invalido["unidad"].iloc[0]
print("Registros de la unidad antes:", (df_limpio["unidad"] == unidad_afectada).sum())

df_limpio = df_limpio[df_limpio["puesto"] != "#N/D"].reset_index(drop=True)

print("Registros de la unidad después:", (df_limpio["unidad"] == unidad_afectada).sum())
print("Registros con #N/D después:", (df_limpio["puesto"] == "#N/D").sum())
print("Total de registros después:", len(df_limpio))

Registros de la unidad antes: 14
Registros de la unidad después: 13
Registros con #N/D después: 0
Total de registros después: 1948


#### Interpretación

Se eliminó un registro y el conjunto de datos pasó de 1,949 a 1,948 registros. El Hospital de Osa no
desaparece del análisis: conserva 13 jefaturas con puestos válidos. La cantidad de puestos distintos
baja de 159 a 158 porque #N/D dejaba de contarse como un puesto.

## 3.5 Anomalías que se conservan

No todo lo que llama la atención en la exploración es un error. Los siguientes casos se revisaron y se
decidió no modificarlos:

| Caso | Registros | Decisión | Justificación |
|---|---|---|---|
| Nombre repetido (JOHNNY MORA QUESADA) | 2 | Se conservan | Tienen cédulas distintas; se trata de dos personas con el mismo nombre |
| Cédulas de 11 dígitos | 8 | Se conservan | Tienen formato válido; corresponden probablemente a documentos de personas extranjeras |
| Puestos sin un término de jefatura (por ejemplo Enfermera 4 Licenciada) | 645 | Se conservan | Son registros reales de la fuente; probablemente indican el puesto en propiedad de la persona que ejerce la jefatura |
| Unidades con una sola jefatura | 89 unidades | Se conservan | Son unidades reales, principalmente sucursales |
| Unidades con muchas jefaturas (valores atípicos) | 27 unidades | Se conservan | Son unidades reales de gran tamaño, principalmente hospitales; su efecto se controla con la transformación logarítmica en el preprocesamiento |
| Diferencias de escritura (Area sin tilde, Farmaceutico, abreviaturas como Hosp.) | Varios | Se conservan | Son los nombres oficiales de la fuente y no generan categorías duplicadas |

In [35]:
cedulas_11 = df_limpio[df_limpio["cedula"].str.len() == 11]
print("Cédulas de 11 dígitos que se conservan:", len(cedulas_11))

nombre_repetido = df[df["Nombre"] == "JOHNNY MORA QUESADA"][["Cedula", "Nombre", "Detalle de UP"]]
nombre_repetido

Cédulas de 11 dígitos que se conservan: 8


,Cedula,Nombre,Detalle de UP
96,107900729,JOHNNY MORA QUESADA,Dirección Inspección
98,106650890,JOHNNY MORA QUESADA,Dirección Inspección


## 3.6 Verificación final

Antes de guardar el resultado se comprueba que el conjunto de datos quedó completo y consistente.

In [36]:
print("Registros:", len(df_limpio))
print("Columnas:", df_limpio.columns.tolist())
print()
print("Valores nulos por columna:")
print(df_limpio.isna().sum())
print()
print("Filas duplicadas:", df_limpio.duplicated().sum())
print("Cédulas repetidas:", df_limpio["cedula"].duplicated().sum())
print()
print("Unidades distintas (código):", df_limpio["codigo_unidad"].nunique())
print("Unidades distintas (nombre):", df_limpio["unidad"].nunique())
print("Puestos distintos:", df_limpio["puesto"].nunique())

Registros: 1948
Columnas: ['cedula', 'codigo_unidad', 'unidad', 'puesto']

Valores nulos por columna:
cedula           0
codigo_unidad    0
unidad           0
puesto           0
dtype: int64

Filas duplicadas: 0
Cédulas repetidas: 0

Unidades distintas (código): 284
Unidades distintas (nombre): 284
Puestos distintos: 158


In [37]:
codigos_con_varios_nombres = (df_limpio.groupby("codigo_unidad")["unidad"].nunique() > 1).sum()
print("Códigos de unidad con más de un nombre:", codigos_con_varios_nombres)

print("Puestos con espacios al inicio o al final:", (df_limpio["puesto"] != df_limpio["puesto"].str.strip()).sum())
print("Puestos con espacios dobles:", df_limpio["puesto"].str.contains("  ").sum())
print("Puestos con #N/D:", (df_limpio["puesto"] == "#N/D").sum())

Códigos de unidad con más de un nombre: 0
Puestos con espacios al inicio o al final: 0
Puestos con espacios dobles: 0
Puestos con #N/D: 0


In [38]:
df_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 1948 entries, 0 to 1947
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   cedula         1948 non-null   str  
 1   codigo_unidad  1948 non-null   str  
 2   unidad         1948 non-null   str  
 3   puesto         1948 non-null   str  
dtypes: str(4)
memory usage: 61.0 KB


In [39]:
df_limpio.head(10)

,cedula,codigo_unidad,unidad,puesto
0,107790683,1100,Gerencia General,Jefe Despacho Gerencia General
1,113570358,1101,Junta Directiva,Jefe Subárea de Sede
2,113750446,1101,Junta Directiva,Jefe Área de Sede
3,107960815,1102,Presidencia Ejecutiva,Asesor Coordinar Presidencia Ejecutiva
4,108340101,1103,Gerencia Financiera,Jefe Área de Sede
5,107300517,1103,Gerencia Financiera,Gerente Financiero
6,109010988,1103,Gerencia Financiera,Jefe Subárea de Sede
7,106990802,1104,Gerencia Administrativa,Jefe Centro de Gestión Informatica Gerencial
8,108860834,1104,Gerencia Administrativa,Gerente Administrativo
9,113680331,1104,Gerencia Administrativa,Analista en Sistemas 4 en TIC


#### Interpretación

El conjunto limpio tiene 1,948 registros y 4 columnas, sin valores nulos, sin filas duplicadas y sin
cédulas repetidas. Se mantienen las 284 unidades y la relación uno a uno entre el código y el nombre de
cada unidad. No quedan espacios sobrantes ni valores inválidos en puesto, y las columnas de
identificación quedaron como texto.

## 3.7 Guardar los datos limpios

El resultado se guarda en data/processed/jefaturas_limpio.csv. Se guarda con codificación UTF-8 y
separado por comas, que es el formato estándar, para que las siguientes etapas lo puedan leer sin
indicar la codificación. Las columnas de código se vuelven a leer como texto para comprobar que el
archivo quedó bien.

In [40]:
ruta_salida = "../data/processed/jefaturas_limpio.csv"

df_limpio.to_csv(ruta_salida, index=False, encoding="utf-8")

comprobacion = pd.read_csv(ruta_salida, dtype={"cedula": str, "codigo_unidad": str})

print("Archivo guardado en:", ruta_salida)
print("Registros leídos:", len(comprobacion))
print("¿Coincide con df_limpio?:", comprobacion.equals(df_limpio))
comprobacion.head()

Archivo guardado en: ../data/processed/jefaturas_limpio.csv
Registros leídos: 1948
¿Coincide con df_limpio?: True


,cedula,codigo_unidad,unidad,puesto
0,107790683,1100,Gerencia General,Jefe Despacho Gerencia General
1,113570358,1101,Junta Directiva,Jefe Subárea de Sede
2,113750446,1101,Junta Directiva,Jefe Área de Sede
3,107960815,1102,Presidencia Ejecutiva,Asesor Coordinar Presidencia Ejecutiva
4,108340101,1103,Gerencia Financiera,Jefe Área de Sede


## 3.8 Resumen de la limpieza

In [41]:
resumen_limpieza = pd.DataFrame([
    ["Selección", "Excluir Fecha de actualización y Nombre", "6 columnas", "4 columnas"],
    ["Formateo", "Renombrar columnas", "Nombres con espacios y tildes", "cedula, codigo_unidad, unidad, puesto"],
    ["Formateo", "Convertir códigos a texto", "cedula y codigo_unidad como int64", "cedula y codigo_unidad como texto"],
    ["Limpieza", "Quitar espacios sobrantes en puesto", f"{len(cambios)} registros con espacios", "0 registros con espacios"],
    ["Limpieza", "Eliminar registro con puesto #N/D", f"{registros_iniciales} registros", f"{len(df_limpio)} registros"],
], columns=["Tarea", "Acción", "Antes", "Después"])

resumen_limpieza

,Tarea,Acción,Antes,Después
0,Selección,Excluir Fecha de actualización y Nombre,6 columnas,4 columnas
1,Formateo,Renombrar columnas,Nombres con espacios y tildes,"cedula, codigo_unidad, unidad, puesto"
2,Formateo,Convertir códigos a texto,cedula y codigo_unidad como int64,cedula y codigo_unidad como texto
3,Limpieza,Quitar espacios sobrantes en puesto,273 registros con espacios,0 registros con espacios
4,Limpieza,Eliminar registro con puesto #N/D,1949 registros,1948 registros


### Conclusión de la limpieza

Los datos originales tenían buena calidad estructural, por lo que la limpieza fue puntual: se
excluyeron dos columnas que no aportan al análisis, se ajustaron los nombres y tipos de las columnas,
se corrigieron los espacios sobrantes en 273 puestos y se eliminó un registro con un valor inválido.
Solo se perdió un registro (0.05 % de los datos).

El resultado es un conjunto de 1,948 jefaturas en 284 unidades, listo para construir los indicadores
por unidad en la segunda parte de esta fase.